In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

artifact_dir = Path("../results/2d-shot-features")
pca_path = artifact_dir / "pca_results.npz"

manifest = json.loads((artifact_dir / "manifest.json").read_text())
metadata = np.load(artifact_dir / "shot_metadata.npz")
pca_artifact = np.load(pca_path)

run_groups = metadata["run_id"]
summary_feature_names = metadata["summary_names"].tolist()
summary_features = metadata["summary_features"]
pca_scores = pca_artifact["scores"]
components = pca_artifact["components"]
physical_pc_deltas = pca_artifact["physical_pc_deltas"]
explained_variance_ratio = pca_artifact["explained_variance_ratio"]
valid_pixels = pca_artifact["valid_pixels"]
representation = str(pca_artifact["representation"])
x_edges_2d = pca_artifact["x_edges"]
y_edges_2d = pca_artifact["y_edges"]

print(f"Loaded {manifest['n_shots']:,} shots from {manifest['n_runs']} runs.")
print(f"Summary features: {summary_feature_names}")
print(f"PCA representation: {representation}")
print(f"PCA scores: {pca_scores.shape}; valid pixels: {valid_pixels.sum():,}")


In [ ]:
n_components_to_plot = min(10, len(components))
extent = [x_edges_2d[0], x_edges_2d[-1], y_edges_2d[0], y_edges_2d[-1]]

def plot_component_maps(values, title, colorbar_label):
    ncols = 5
    nrows = int(np.ceil(len(values) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(18, 3.5 * nrows), sharex=True, sharey=True)
    axs = np.atleast_1d(axs).flat
    for component_index, ax in enumerate(axs):
        if component_index >= len(values):
            ax.axis("off")
            continue
        image = np.full(valid_pixels.shape, np.nan)
        image[valid_pixels] = values[component_index]
        vmax = np.nanmax(np.abs(image))
        im = ax.imshow(image.T, origin="lower", extent=extent, cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(f"PC{component_index + 1}: {explained_variance_ratio[component_index]:.2%}")
        ax.set_xlabel("x [um]")
        ax.set_ylabel("y [um]")
        fig.colorbar(im, ax=ax, shrink=0.75, label=colorbar_label)
    fig.suptitle(title, y=1.01)
    fig.tight_layout()

plot_component_maps(components[:n_components_to_plot], "Standardized-pixel PCA loadings", "loading")
plot_component_maps(physical_pc_deltas[:n_components_to_plot], "Physical +/-1 SD profile changes", "excitation-fraction change")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(explained_variance_ratio) + 1), np.cumsum(explained_variance_ratio), marker="o")
plt.xlabel("number of PCs")
plt.ylabel("cumulative explained variance")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)


In [ ]:
x_centers = 0.5 * (x_edges_2d[:-1] + x_edges_2d[1:])
y_centers = 0.5 * (y_edges_2d[:-1] + y_edges_2d[1:])
x_grid, y_grid = np.meshgrid(x_centers, y_centers, indexing="ij")
x_valid = x_grid[valid_pixels]
y_valid = y_grid[valid_pixels]

mean_count_weights = pca_artifact["mean_valid_counts"]
mean_count_weights = mean_count_weights / mean_count_weights.sum()

def centered(values):
    return values - np.sum(mean_count_weights * values)

templates = {
    "x gradient": centered(x_valid),
    "y gradient": centered(y_valid),
    "x curvature": centered(x_valid**2),
    "xy curvature": centered(x_valid * y_valid),
    "y curvature": centered(y_valid**2),
    "radial curvature": centered(x_valid**2 + y_valid**2),
}
scaler_scale = pca_artifact["scaler_scale"]
standardized_templates = {name: values / scaler_scale for name, values in templates.items()}
standardized_templates = {name: values / np.linalg.norm(values) for name, values in standardized_templates.items()}
overlaps = pd.DataFrame(
    {name: np.abs(components @ template) for name, template in standardized_templates.items()},
    index=[f"PC{i + 1}" for i in range(len(components))],
)
display(overlaps.round(3))
display((overlaps**2).cumsum().round(3))

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(overlaps.values, aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(np.arange(len(overlaps.columns)), overlaps.columns, rotation=35, ha="right")
ax.set_yticks(np.arange(len(overlaps.index)), overlaps.index)
fig.colorbar(im, ax=ax, label="absolute overlap")
fig.tight_layout()


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

n_predictive_pcs = min(5, pca_scores.shape[1])
feature_names = summary_feature_names + [f"PC{i + 1}_score" for i in range(n_predictive_pcs)]
X = np.column_stack([summary_features, pca_scores[:, :n_predictive_pcs]])
targets = {
    "mu_x0": metadata["mu_x0"], "mu_y0": metadata["mu_y0"],
    "sigma_x0": metadata["sigma_x"], "sigma_y0": metadata["sigma_y"],
    "mu_vx0": metadata["mu_vx0"], "mu_vy0": metadata["mu_vy0"],
    "sigma_vx0": metadata["sigma_vx"], "sigma_vy0": metadata["sigma_vy"],
}
quadratic_model = Pipeline([
    ("input_scaler", StandardScaler()),
    ("quadratic_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("quadratic_scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1e-2)),
])

all_data_rows = []
all_data_predictions = {}
for target_name, y in targets.items():
    model = clone(quadratic_model).fit(X, y)
    prediction = model.predict(X)
    all_data_predictions[target_name] = prediction
    all_data_rows.append({"target": target_name, "R2": r2_score(y, prediction), "RMSE": np.sqrt(np.mean((y - prediction)**2))})
all_data_results = pd.DataFrame(all_data_rows).set_index("target")
display(all_data_results)

fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for ax, (target_name, y) in zip(axs.flat, targets.items()):
    prediction = all_data_predictions[target_name]
    lo, hi = min(y.min(), prediction.min()), max(y.max(), prediction.max())
    ax.scatter(y, prediction, alpha=0.15, s=8)
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_title(f"{target_name}: R2={r2_score(y, prediction):.3f}")
    ax.set_xlabel("true")
    ax.set_ylabel("fitted")
fig.tight_layout()


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

unique_runs = np.unique(run_groups)
if len(unique_runs) < 2:
    print("Grouped holdout skipped: at least two complete runs are required.")
else:
    train_index, test_index = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, groups=run_groups))
    holdout_rows = []
    for target_name, y in targets.items():
        model = clone(quadratic_model).fit(X[train_index], y[train_index])
        prediction = model.predict(X[test_index])
        holdout_rows.append({
            "target": target_name,
            "held_out_run_R2": r2_score(y[test_index], prediction),
            "held_out_run_RMSE": np.sqrt(np.mean((y[test_index] - prediction)**2)),
        })
    display(pd.DataFrame(holdout_rows).set_index("target"))
    print(f"Held out {len(np.unique(run_groups[test_index]))} complete runs ({len(test_index):,} shots).")
